# GEAP Multi-Turn Evaluation — without patching the SDK (SDK-First)

The main eval notebook (`evaluation_sdk_demo.ipynb`) is **single-turn**: every case is one prompt → one response. This notebook shows **multi-turn** conversation evaluation — scoring a whole dialogue with the `MULTI_TURN_*` rubric autoraters — using only public SDK APIs (no monkey-patching).

**Why a separate notebook / SDK bump.** The native multi-turn path (`run_inference(agent=..., config=EvalRunInferenceConfig(user_simulator_config=...))`) is **still broken** on `google-cloud-aiplatform` **1.162 and 1.163**: parsing the agent's streamed events into the `types.evals.AgentData` model raises *`extra_forbidden`* (`model_version`, `content`, …). So instead we **generate the conversation ourselves** (a real threaded Agent Engine session) and hand it to the scorer as an `EvalCase` — which works cleanly. Requires **aiplatform ≥ 1.163** for the `MULTI_TURN_*` metrics + `EvalCase.conversation_history` (the workshop pins 1.162; bump to enable this).

**Legend.** ✅ runs live. 🔧 acknowledged custom (not the eval SDK). Scoring itself is 100% L1 SDK (`client.evals.evaluate` + `types.RubricMetric.MULTI_TURN_*`).

## Setup

In [1]:
import os
for _ in range(6):
    if os.path.exists("src/config.py"):
        break
    os.chdir("..")

import importlib.metadata as _md
_ver = tuple(int(x) for x in _md.version("google-cloud-aiplatform").split(".")[:2])
assert _ver >= (1, 163), f"multi-turn needs aiplatform>=1.163 (found {_md.version('google-cloud-aiplatform')})"

import vertexai
from vertexai import Client, types, agent_engines
from vertexai.types import evals as ev
from google.genai import types as g
from src.config import GCP_PROJECT_ID, GCP_REGION, AGENT_ENGINE_ID

vertexai.init(project=GCP_PROJECT_ID, location=GCP_REGION)
client = Client(project=GCP_PROJECT_ID, location=GCP_REGION)
AGENT_RESOURCE = f"projects/{GCP_PROJECT_ID}/locations/{GCP_REGION}/reasoningEngines/{AGENT_ENGINE_ID}"
engine = agent_engines.get(AGENT_RESOURCE)

def content(role, text):
    return g.Content(role=role, parts=[g.Part.from_text(text=text)])

print("aiplatform:", _md.version("google-cloud-aiplatform"), "| agent:", AGENT_RESOURCE)

/tmp/ipykernel_4044728/1252865920.py:18: FutureWarning: The vertexai.Client class is deprecated. Please use agentplatform.Client instead.
  client = Client(project=GCP_PROJECT_ID, location=GCP_REGION)


aiplatform: 1.163.0 | agent: projects/wortz-project-352116/locations/us-central1/reasoningEngines/5895016748914049024


## Phase 1 — Drive a real multi-turn conversation — 🔧 custom

To get a genuine multi-turn trace we hold **one Agent Engine session** across turns (`create_session` + `stream_query(session_id=...)`), so the agent remembers earlier turns. Each later prompt depends on the previous answer.

In [2]:
user_id = "multi-turn-demo"
session_id = engine.create_session(user_id=user_id)["id"]

def ask(message: str) -> str:
    """🔧 One turn on the shared session; returns the agent's final text."""
    parts = [p["text"] for e in engine.stream_query(message=message, user_id=user_id, session_id=session_id)
             for p in ((e.get("content") or {}).get("parts") or []) if p.get("text")]
    return "\n".join(parts).strip() or "(no response)"

dialogue = [
    "What is the corporate meal expense limit?",
    "Is a $90 dinner within that policy?",   # depends on turn 1
]
transcript = []
for turn in dialogue:
    answer = ask(turn)
    transcript.append((turn, answer))
    print(f"USER : {turn}\nAGENT: {answer[:160]}\n")

USER : What is the corporate meal expense limit?
AGENT: The corporate meal expense limit is $75.



USER : Is a $90 dinner within that policy?
AGENT: No, a $90 dinner is not within policy. The limit for meals is $75, so a $90 dinner exceeds that by $15. This would require manager review.



## Phase 2 — Assemble a multi-turn `EvalCase`
📖 [evaluate-agents](https://docs.cloud.google.com/gemini-enterprise-agent-platform/optimize/evaluation/evaluate-agents)

The `MULTI_TURN_*` metrics read three fields: **`conversation_history`** (the prior turns, as `ev.Message`), **`prompt`** (the final user turn), and **`responses`** (the final agent answer being scored). The autorater judges that last answer in the context of the whole dialogue. All public types — nothing patched.

In [3]:
def multi_turn_case(transcript) -> types.EvalCase:
    """Prior turns -> conversation_history; last turn -> prompt + responses."""
    history = []
    for user_msg, agent_msg in transcript[:-1]:
        history.append(ev.Message(author="user",  content=content("user",  user_msg)))
        history.append(ev.Message(author="model", content=content("model", agent_msg)))
    last_user, last_answer = transcript[-1]
    return types.EvalCase(
        conversation_history=history,
        prompt=content("user", last_user),
        responses=[types.ResponseCandidate(response=content("model", last_answer))],
    )

case = multi_turn_case(transcript)
print("conversation_history turns:", len(case.conversation_history), "| scoring final answer of", len(transcript), "turns")

conversation_history turns: 2 | scoring final answer of 2 turns


## Phase 3 — Score with `MULTI_TURN_*` rubric metrics
📖 [manage-metrics](https://docs.cloud.google.com/gemini-enterprise-agent-platform/optimize/evaluation/manage-metrics)

Multi-turn autoraters analyze the full conversation: **task success** (were the goals met across turns?), **general quality**, and **trajectory quality** (was the reasoning path coherent?). Pure L1 SDK.

> Reading the scores: `general_quality` and `trajectory_quality` rate the dialogue on its own terms; `task_success` is low for a short **informational** Q&A because there's no completable task — swap in a task-oriented dialogue (e.g. *search flights → book the first one*) to exercise it. That contrast is exactly what the three multi-turn metrics are for.

In [4]:
def show_summary(result, threshold: float = 3.0):
    for m in (result.summary_metrics or []):
        mean = m.mean_score or 0.0
        score = mean * 5 if mean <= 1.0 else mean   # SDK 0-1 -> repo 1-5 convention
        flag = "PASS" if score >= threshold else "FAIL"
        print(f"  {m.metric_name:34s} {score:4.2f}/5  [{flag}]  (errors={m.num_cases_error}/{m.num_cases_total})")

result = client.evals.evaluate(
    dataset=types.EvaluationDataset(eval_cases=[case]),
    metrics=[types.RubricMetric.MULTI_TURN_TASK_SUCCESS,
             types.RubricMetric.MULTI_TURN_GENERAL_QUALITY,
             types.RubricMetric.MULTI_TURN_TRAJECTORY_QUALITY],
)
show_summary(result)

14:56:15 - LiteLLM:WARNING: common_utils.py:979 - litellm: could not pre-load bedrock-runtime response stream shape — Bedrock event-stream decoding will be unavailable. Error: No module named 'botocore'


14:56:15 - LiteLLM:WARNING: common_utils.py:24 - litellm: could not pre-load sagemaker-runtime response stream shape — SageMaker event-stream decoding will be unavailable. Error: No module named 'botocore'


Computing Metrics for Evaluation Dataset:   0%|          | 0/3 [00:00<?, ?it/s]

Computing Metrics for Evaluation Dataset:  33%|███▎      | 1/3 [00:26<00:53, 26.71s/it]

Computing Metrics for Evaluation Dataset:  67%|██████▋   | 2/3 [01:40<00:54, 54.69s/it]

Retryable error (code=500) on attempt 1/5 for metric 'multi_turn_trajectory_quality_v1': 500 INTERNAL. {'error': {'code': 500, 'message': 'Internal error encountered.', 'status': 'INTERNAL'}}. Retrying in 1.5 seconds...


Computing Metrics for Evaluation Dataset: 100%|██████████| 3/3 [03:50<00:00, 88.94s/it]

Computing Metrics for Evaluation Dataset: 100%|██████████| 3/3 [03:50<00:00, 76.90s/it]

  multi_turn_task_success_v1         0.00/5  [FAIL]  (errors=0/1)
  multi_turn_general_quality_v1      5.00/5  [PASS]  (errors=0/1)
  multi_turn_trajectory_quality_v1   5.00/5  [PASS]  (errors=0/1)


## What does NOT work without patching (on 1.162 / 1.163)

The **native user-simulator** path — `client.evals.run_inference(agent=..., config=types.EvalRunInferenceConfig(user_simulator_config=ev.UserSimulatorConfig(model_name=..., max_turn=N)))` — raises `ValidationError: N validation errors for AgentData ... extra_forbidden (model_version, content, ...)` because the SDK parses the agent's raw stream events into the strict `AgentData` model. That would require patching the SDK's `AgentData`/`AgentEvent` models (which we do **not** do). Generating the transcript ourselves (Phase 1) and scoring an `EvalCase` (Phase 2–3) sidesteps it entirely.

## Recap
- **Multi-turn eval works patch-free on aiplatform ≥ 1.163**: thread a real session (`create_session` + `stream_query`), build `EvalCase(conversation_history + prompt + responses)`, score with `types.RubricMetric.MULTI_TURN_*`.
- **Broken (both 1.162 and 1.163, needs a patch to use):** the native `run_inference` user-simulator (`AgentData` `extra_forbidden`).
- To adopt: bump the workshop pin from `1.162` to `≥ 1.163` (single-turn behavior is unchanged).